# High-Resolution Transmission Electron Microscopy (HRTEM) Simulation

High-resolution transmission electron microscopy (HRTEM) forms phase-contrast images
of thin specimens at atomic resolution. Because electrons are strongly scattered by
electrostatic crystal potentials, an HRTEM micrograph is not a simple absorption image;
it is the interference pattern of transmitted and diffracted electron waves modulated
by the coherent transfer characteristics and aberrations of the objective lens.

This tutorial demonstrates the complete HRTEM simulation pipeline in PyTex:

1. **The Objective Lens Contrast Transfer Function (CTF)**: wave aberration function $\chi(q)$,
   Scherzer underfocus, $C_s$-correction, and double correction ($C_s + C_c$).
2. **Phase Contrast Regimes**: comparing conventional Scherzer dark atomic columns with
   Negative Spherical Aberration Imaging (NCSI) bright atomic columns.
3. **Crystalline Supercells and Point Defects**: modeling vacancy defects in an FCC lattice
   and observing local intensity contrast signatures.
4. **Volterra Edge Dislocations**: introducing lattice displacement fields to observe atomic
   column distortion at a dislocation core.
5. **Amorphous Foils and Thon Rings**: simulating disordered carbon foils and reading 2D FFT
   power spectrum rings for aberration diagnosis.
6. **Multislice Engine Integration**: interfacing with `abtem` and generating convention-explicit
   scientific descriptions.

The theoretical foundations are detailed in {doc}`../../theory/hrem_multislice_and_ctf`.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from pytex import FrameDomain, Handedness, Lattice, Phase, ReferenceFrame, SpaceGroupSpec, SymmetrySpec, UnitCell, AtomicSite
from pytex.diffraction.hrem import (
    AtomicSnapshot,
    DoubleCorrectionMode,
    MicroscopeAberrations,
    pure_python_phase_object_simulation,
    relativistic_interaction_parameter_inv_v_angstrom,
    relativistic_wavelength_angstrom,
)
from pytex.adapters.abtem import is_abtem_available, simulate_hrem

plt.rcParams["figure.dpi"] = 110
crystal = ReferenceFrame("crystal", FrameDomain.CRYSTAL, ("a", "b", "c"), Handedness.RIGHT)


## 1. Objective Lens Optics and Contrast Transfer Function

The coherent contrast transfer function (CTF) of the objective lens describes how spatial
frequencies $q$ in the exit wave are transmitted to the image plane. The wave aberration
function $\chi(q)$ accounts for defocus $\Delta f$ and spherical aberration $C_s$:

$$\chi(q) = \pi \Delta f \lambda q^2 + \frac{1}{2} \pi C_s \lambda^3 q^4 + \frac{1}{3} \pi C_5 \lambda^5 q^6$$

Under partial coherence, high spatial frequencies are damped by the temporal envelope $E_c(q)$
(focal spread $\Delta$, dominated by chromatic aberration $C_c$ and energy spread $\Delta E$)
and the spatial envelope $E_s(q)$ (convergence semiangle $\alpha_s$):

$$E_c(q) = \exp\left(-\frac{1}{2} \pi^2 \lambda^2 \Delta^2 q^4\right)$$
$$E_s(q) = \exp\left(-\pi^2 \alpha_s^2 (\Delta f q + C_s \lambda^2 q^3)^2\right)$$

In conventional uncorrected TEM ($C_s > 0$), Scherzer defocus balances defocus against $C_s$:
$$\Delta f_{\mathrm{Sch}} = -1.2\sqrt{C_s \lambda}$$
yielding a broad passband of constant phase contrast up to the Scherzer point resolution:
$$d_{\mathrm{Sch}} = 0.64 (C_s \lambda^3)^{1/4}$$

In a modern double-corrected microscope ($C_s$- and $C_c$-corrected), both $C_s$ and focal spread
$\Delta$ are suppressed, extending the information limit well into the sub-angstrom regime.


In [ ]:
# 1. Uncorrected conventional TEM (200 kV, Cs = 1.0 mm)
uncorrected = MicroscopeAberrations.conventional_tem(
    energy_kev=200.0,
    cs_mm=1.0,
    focal_spread_angstrom=35.0,
    convergence_semiangle_mrad=0.5,
)

# 2. Cs-corrected TEM (200 kV, Cs = 5 µm)
cs_corr = MicroscopeAberrations.cs_corrected(
    energy_kev=200.0,
    cs_um=5.0,
    defocus_angstrom=-25.0,
    focal_spread_angstrom=20.0,
    convergence_semiangle_mrad=0.2,
)

# 3. Double-corrected TEM (Cs + Cc corrected, 300 kV, Cs = 0, Delta = 5 Å)
double_corr = MicroscopeAberrations.double_corrected(
    energy_kev=300.0,
    cs_um=0.0,
    defocus_angstrom=-10.0,
    focal_spread_angstrom=5.0,
    convergence_semiangle_mrad=0.1,
)

ctf_uncorr = uncorrected.evaluate_ctf_1d(max_q_inv_angstrom=1.2, num_points=500)
ctf_cs = cs_corr.evaluate_ctf_1d(max_q_inv_angstrom=1.5, num_points=500)
ctf_double = double_corr.evaluate_ctf_1d(max_q_inv_angstrom=2.5, num_points=500)

fig, axes = plt.subplots(3, 1, figsize=(9, 7.5), sharex=False)

# Uncorrected
axes[0].plot(ctf_uncorr.q_inv_angstrom, ctf_uncorr.transfer, "b-", lw=1.5, label="CTF")
axes[0].plot(ctf_uncorr.q_inv_angstrom, ctf_uncorr.total_envelope, "k:", alpha=0.7, label=r"$E_{\mathrm{tot}}(q)$")
axes[0].axhline(0, color="gray", lw=0.5)
if not np.isnan(ctf_uncorr.point_resolution_angstrom):
    axes[0].axvline(1.0 / ctf_uncorr.point_resolution_angstrom, color='red', linestyle='--',
                    label=f'Scherzer cutoff ({ctf_uncorr.point_resolution_angstrom:.2f} Å)')
axes[0].set_title(f'Uncorrected TEM (200 kV, Cs=1.0 mm, Δf={uncorrected.defocus_angstrom:.1f} Å)')
axes[0].set_ylabel("Transfer")
axes[0].legend(loc="upper right", fontsize=8)
axes[0].set_ylim(-1.1, 1.1)

# Cs-corrected
axes[1].plot(ctf_cs.q_inv_angstrom, ctf_cs.transfer, "darkgreen", lw=1.5, label="CTF")
axes[1].plot(ctf_cs.q_inv_angstrom, ctf_cs.total_envelope, "k:", alpha=0.7, label=r"$E_{\mathrm{tot}}(q)$")
axes[1].axhline(0, color="gray", lw=0.5)
axes[1].set_title(f'Cs-Corrected TEM (200 kV, Cs=5 µm, Δf={cs_corr.defocus_angstrom:.1f} Å)')
axes[1].set_ylabel("Transfer")
axes[1].legend(loc="upper right", fontsize=8)
axes[1].set_ylim(-1.1, 1.1)

# Double-corrected
axes[2].plot(ctf_double.q_inv_angstrom, ctf_double.transfer, "purple", lw=1.5, label="CTF")
axes[2].plot(ctf_double.q_inv_angstrom, ctf_double.total_envelope, "k:", alpha=0.7, label=r"$E_{\mathrm{tot}}(q)$")
axes[2].axhline(0, color="gray", lw=0.5)
axes[2].set_title(f'Double-Corrected TEM (Cs+Cc, 300 kV, Cs=0, Δ={double_corr.focal_spread_angstrom:.1f} Å)')
axes[2].set_xlabel("Spatial frequency q (1/Å)")
axes[2].set_ylabel("Transfer")
axes[2].legend(loc="upper right", fontsize=8)
axes[2].set_ylim(-1.1, 1.1)

fig.tight_layout()
plt.show()


## 2. Oriented Crystalline Supercell Creation

We define an FCC Aluminum crystal ($a = 4.0495\text{ Å}$, space group $Fm\bar{3}m$, No. 225)
and construct an oriented supercell along the $[001]$ zone axis using `AtomicSnapshot.from_phase`.


In [ ]:
al_lattice = Lattice(4.0495, 4.0495, 4.0495, 90.0, 90.0, 90.0, crystal_frame=crystal)
aluminum = Phase(
    name="aluminum-fcc",
    lattice=al_lattice,
    symmetry=SymmetrySpec.from_point_group("m-3m", reference_frame=crystal),
    crystal_frame=crystal,
    unit_cell=UnitCell(
        lattice=al_lattice,
        sites=tuple(
            AtomicSite(
                label=f"Al{i}",
                species="Al",
                fractional_coordinates=np.asarray(pos, dtype=float),
            )
            for i, pos in enumerate(
                [(0.0, 0.0, 0.0), (0.0, 0.5, 0.5), (0.5, 0.0, 0.5), (0.5, 0.5, 0.0)]
            )
        ),
    ),
    space_group=SpaceGroupSpec(symbol="Fm-3m", number=225, reference_frame=crystal),
)

# Build 4 x 4 x 2 supercell snapshot (~16.2 x 16.2 x 8.1 Å)
al_snapshot = AtomicSnapshot.from_phase(
    phase=aluminum,
    supercell=(4, 4, 2),
    zone_axis=(0, 0, 1),
    label="Al [001] supercell",
)
print(f"Created Al supercell: {al_snapshot.natoms} atoms, box dimensions = {al_snapshot.dimensions_angstrom} Å")


## 3. Imaging Regimes: Scherzer vs Double-Corrected vs NCSI

HRTEM images change dramatically depending on the lens aberrations and defocus:

* **Scherzer underfocus**: with positive $C_s$, $\sin\chi(q) < 0$ across the passband.
  Atomic columns appear as **dark minima** on a bright background (negative phase contrast).
* **Double-corrected TEM**: residual $C_s \approx 0$, no passband oscillation out to
  high spatial frequencies, giving clean delineation of atomic positions.
* **Negative Spherical Aberration Imaging (NCSI)**: with $C_s < 0$ (e.g. $-15\ \mu\mathrm{m}$)
  and slight overfocus ($\Delta f > 0$, e.g. $+50\text{ Å}$), $\sin\chi(q) > 0$ across the passband.
  Atomic columns appear as **bright maxima** on a dark background (positive phase contrast).


In [ ]:
# NCSI optical setup
ncsi_optics = MicroscopeAberrations.ncsi(energy_kev=200.0)

# Simulate with multislice / phase-object engine
sim_scherzer = simulate_hrem(al_snapshot, uncorrected, sampling_angstrom=0.08)
sim_double = simulate_hrem(al_snapshot, double_corr, sampling_angstrom=0.08)
sim_ncsi = simulate_hrem(al_snapshot, ncsi_optics, sampling_angstrom=0.08)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
lx, ly = al_snapshot.dimensions_angstrom[:2]

axes[0].imshow(sim_scherzer.image, cmap="gray", origin="lower", extent=(0, lx, 0, ly))
axes[0].set_title(f"Scherzer (Dark Atoms)\nContrast: {sim_scherzer.michelson_contrast*100:.1f}%")
axes[0].set_xlabel("x (Å)")
axes[0].set_ylabel("y (Å)")

axes[1].imshow(sim_double.image, cmap="gray", origin="lower", extent=(0, lx, 0, ly))
axes[1].set_title(f"Double-Corrected\nContrast: {sim_double.michelson_contrast*100:.1f}%")
axes[1].set_xlabel("x (Å)")

axes[2].imshow(sim_ncsi.image, cmap="gray", origin="lower", extent=(0, lx, 0, ly))
axes[2].set_title(f"NCSI (Bright Atoms)\nContrast: {sim_ncsi.michelson_contrast*100:.1f}%")
axes[2].set_xlabel("x (Å)")

fig.tight_layout()
plt.show()


## 4. Point Defect Simulation: Atomic Vacancies

Point defects alter the local projected electrostatic potential $V_p(x, y)$, modifying the
phase shift $\sigma V_p(x, y)$ experienced by the electron wave. Using
`AtomicSnapshot.crystalline_with_vacancy`, we engineer a vacancy column in the supercell
and simulate its HRTEM signature.


In [ ]:
al_vacancy = AtomicSnapshot.crystalline_with_vacancy(
    phase=aluminum,
    supercell=(5, 5, 2),
    vacancy_count=1,
)

sim_vacancy = simulate_hrem(al_vacancy, double_corr, sampling_angstrom=0.08)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.5))
extent = (0, sim_vacancy.extent_angstrom[0], 0, sim_vacancy.extent_angstrom[1])
ax1.imshow(sim_vacancy.image, cmap="gray", origin="lower", extent=extent)
ax1.set_title(f"Double-Corrected Al HRTEM with Vacancy\n({al_vacancy.natoms} atoms)")
ax1.set_xlabel("x (Å)")
ax1.set_ylabel("y (Å)")

# Line profile through central vacancy row
ny, nx = sim_vacancy.image.shape
center_y = ny // 2
x_coords = np.linspace(0, sim_vacancy.extent_angstrom[0], nx)
profile = sim_vacancy.image[center_y, :]

ax2.plot(x_coords, profile, "b.-", lw=1.5)
ax2.set_title("Intensity Profile across Defect Row")
ax2.set_xlabel("x (Å)")
ax2.set_ylabel("Normalized Intensity")
ax2.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()


## 5. Volterra Dislocation Core Simulation

Lattice dislocations produce long-range elastic strain fields governed by continuum elasticity.
Using `AtomicSnapshot.crystalline_with_dislocation`, we apply the Volterra displacement field
for an edge dislocation ($b = 2.86\text{ Å}$) with its line vector along the electron beam.


In [ ]:
al_disloc = AtomicSnapshot.crystalline_with_dislocation(
    phase=aluminum,
    supercell=(6, 6, 2),
    burgers_vector_angstrom=2.86,
    dislocation_type="edge",
)

sim_disloc = simulate_hrem(al_disloc, cs_corr, sampling_angstrom=0.08)

fig, ax = plt.subplots(figsize=(5.5, 5.5))
extent = (0, sim_disloc.extent_angstrom[0], 0, sim_disloc.extent_angstrom[1])
ax.imshow(sim_disloc.image, cmap="gray", origin="lower", extent=extent)
ax.set_title(f"Edge Dislocation Core in Al\nb = 2.86 Å, {al_disloc.natoms} atoms")
ax.set_xlabel("x (Å)")
ax.set_ylabel("y (Å)")
fig.tight_layout()
plt.show()


## 6. Amorphous Foil Simulation and Thon Rings

Amorphous carbon foils exhibit random atomic arrangements whose Fourier power spectrum is
approximately white noise. When modulated by the CTF, the 2D power spectrum displays concentric
rings known as **Thon rings**:

$$P(\mathbf{q}) = |\mathcal{F}\{I(\mathbf{r})\}|^2 \propto |E_{\mathrm{tot}}(q)|^2 \sin^2 \chi(q)$$

The ring radii directly map the zero crossings of $\sin\chi(q)$ and allow experimentalists to
measure defocus $\Delta f$ and diagnose astigmatism.


In [ ]:
carbon_foil = AtomicSnapshot.amorphous_sample(
    species="C",
    density_g_cm3=2.0,
    dimensions_angstrom=(35.0, 35.0, 15.0),
    seed=101,
)

amorphous_optics = MicroscopeAberrations(
    energy_kev=200.0,
    cs_mm=1.0,
    defocus_angstrom=-600.0,
    focal_spread_angstrom=25.0,
    convergence_semiangle_mrad=0.3,
)

sim_amorphous = simulate_hrem(carbon_foil, amorphous_optics, sampling_angstrom=0.08)

fig, (ax_img, ax_fft) = plt.subplots(1, 2, figsize=(10, 4.5))
extent_img = (0, sim_amorphous.extent_angstrom[0], 0, sim_amorphous.extent_angstrom[1])
ax_img.imshow(sim_amorphous.image, cmap="gray", origin="lower", extent=extent_img)
ax_img.set_title(f"Amorphous Carbon Speckle\n({carbon_foil.natoms} atoms, {carbon_foil.thickness_angstrom:.1f} Å thick)")
ax_img.set_xlabel("x (Å)")
ax_img.set_ylabel("y (Å)")

max_q = 0.5 / sim_amorphous.pixel_size_angstrom
ax_fft.imshow(
    sim_amorphous.power_spectrum,
    cmap="inferno",
    origin="lower",
    extent=(-max_q, max_q, -max_q, max_q),
)
ax_fft.set_title(r"2D FFT Power Spectrum: Thon Rings ($\sin^2\chi$)")
ax_fft.set_xlabel("Spatial frequency qx (1/Å)")
ax_fft.set_ylabel("Spatial frequency qy (1/Å)")
ax_fft.set_xlim(-1.2, 1.2)
ax_fft.set_ylim(-1.2, 1.2)

fig.tight_layout()
plt.show()


## 7. Explainable Scientific Output

In accordance with PyTex's explainable-results doctrine, every result object provides
a `.describe()` method returning human-readable scientific prose documenting the parameters,
resolutions, and physical findings.


In [ ]:
description = sim_vacancy.describe()
print(description[:120] + '...')
